# 04 · Agente: estado, herramientas, validaciones y aclaraciones

**Laboratorio 4 · Asistente de compras por voz** (MCD8016 Deep Learning, UTEC)

El enunciado pide un agente que *"seleccione las operaciones necesarias según la solicitud y los resultados obtenidos"*, que mantenga *"el estado de la conversación"*, que solicite *"aclaraciones cuando falte información o exista una ambigüedad relevante"* y que valide *"cantidades, presupuesto y existencias mediante código"*. Además, *"el control del agente, la memoria, las herramientas y las validaciones deberán ser implementados por el grupo"*: aquí no se usa ningún framework de agentes (LangChain, etc.).

### Diseño: el LLM decide, el código valida

```
 usuario ──► [ estado (código) ] ──► prompt del sistema + ESTADO ACTUAL
                                          │
                                   ┌──────▼──────┐   tool call (JSON)   ┌────────────────────┐
                                   │     LLM     │ ───────────────────► │ herramientas       │
                                   │ (Qwen, 4bit)│ ◄─────────────────── │ valida → tienda/RAG│
                                   └──────┬──────┘   resultado (JSON)   └────────────────────┘
                                          │ respuesta final
                                   [ control de afirmaciones (código) ] ──► usuario
```

| Pieza | Implementación | Qué garantiza |
|---|---|---|
| **Estado** | `ConversationState`: presupuesto, preferencias, refs mostradas, carrito, cotización y pedido | Se re-inyecta en cada llamada; no depende de la memoria del LLM |
| **Herramientas** | 9 funciones con esquema JSON (`TOOL_SCHEMAS`) | Cada una valida tipos, refs, cantidades, stock y presupuesto |
| **Regla de confirmación** | `confirmar_pedido` exige que la cotización se haya mostrado en un turno **anterior** | Ningún pedido sin confirmación explícita del usuario |
| **Control de afirmaciones** | `check_response` revisa la respuesta final | El LLM no puede decir "agregué" o "pedido registrado" si no ocurrió |
| **Refs cortas** (R1, R2…) | El LLM nunca copia ids de Amazon de 10 caracteres | Evita errores de transcripción de ids |
| **Bucle acotado** | Máximo 8 pasos y 2 correcciones por turno | Sin bucles infinitos; respuesta de respaldo si falla |

Código en `src/asistente_compras/agent.py` y `llm.py`; pruebas en `tests/test_agent.py`.

## 0. Entorno

In [ ]:
# ── Configuración del entorno (idéntica en todos los notebooks) ──────────────────────────
# Detectar si corre en Colab o en la Mac y dejar importable el paquete `asistente_compras`.
import os
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    # 1) Montar Drive: guarda el código y los datos entre sesiones (Colab borra /content al cerrar).
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/lab4/lab4_solution")
    os.environ["LAB4_DATA_DIR"] = "/content/drive/MyDrive/lab4/data"
    # 2) Instalar el paquete y sus dependencias. EXTRAS cambia según el notebook
    #    ("" = núcleo; "rag", "llm", "voz" en los siguientes).
    EXTRAS = "rag,llm"
    target = f"{PROJECT_DIR}[{EXTRAS}]" if EXTRAS else str(PROJECT_DIR)
    get_ipython().run_line_magic("pip", f"install -q -e {target}")
else:
    # En la Mac, usar como kernel el entorno uv del proyecto (ver README).
    PROJECT_DIR = Path.cwd().resolve().parent

# Una instalación editable recién hecha no se ve hasta reiniciar el kernel;
# agregar src/ al path para evitar ese reinicio.
sys.path.insert(0, str(PROJECT_DIR / "src"))
print(f"Colab: {EN_COLAB} | proyecto: {PROJECT_DIR}")

In [ ]:
import pandas as pd

from asistente_compras.agent import (
    TOOL_SCHEMAS, ConversationState, ShoppingAgent, ShoppingTools, format_turn,
)
from asistente_compras.config import get_paths
from asistente_compras.evaluation import timed
from asistente_compras.llm import MODELS, LLMReply, make_llm
from asistente_compras.retrieval import build_system_retriever
from asistente_compras.store import Store, restore_store

# Elegir el modelo del agente. En la Mac corre vía Ollama y en Colab vía transformers 4-bit (mismo modelo).
# Justificar la elección final con los 20 escenarios del notebook 05.
MODELO = "qwen3.5:9b"  # mejor conducta en la Mac; la elección final se hace en Colab

paths = get_paths().ensure()
catalog = pd.read_parquet(paths.processed / "catalog.parquet")
retriever, t_ret = timed(build_system_retriever, catalog, paths.index)
llm, t_llm = timed(make_llm, MODELO)
print(f"Recuperador: {t_ret:.1f} s · LLM {llm.name}: {t_llm:.1f} s (carga inicial)")

## 1. Las herramientas

Son lo único que el LLM puede hacer sobre el mundo. Los esquemas siguen el formato de *function calling* y se insertan en el prompt con la plantilla nativa de Qwen, que es la misma en Ollama y en `transformers`.

In [ ]:
pd.DataFrame([{"herramienta": t["function"]["name"],
               "argumentos": ", ".join(t["function"]["parameters"]["properties"]) or "—",
               "descripción": t["function"]["description"]} for t in TOOL_SCHEMAS])

### 1.1 Validaciones, sin LLM
Las herramientas se llaman directamente para mostrar que las reglas viven en el código:
- Un atributo ausente en la ficha se devuelve como **"desconocido"**, para que el agente no lo invente.
- Una cantidad no entera o una ref inexistente se rechazan antes de tocar la tienda.
- Precio y stock salen de la tienda en el momento, nunca del índice.

In [ ]:
restore_store(paths.store_db, catalog.product_id)
store = Store(paths.store_db)
tools = ShoppingTools(store, retriever, catalog, ConversationState())

r = tools.buscar_productos("funda iphone 13", precio_max_eur=15)
pd.DataFrame(r["resultados"])[["ref", "nombre_corto", "marca", "precio", "stock"]]

In [ ]:
# Ver la ficha completa: los campos vacíos del catálogo quedan como "desconocido".
{k: (v[:80] + "…" if isinstance(v, str) and len(v) > 80 else v)
 for k, v in tools.ver_detalle("R1").items()}

In [ ]:
for args in [{"ref": "R1", "cantidad": 2.5}, {"ref": "R9", "cantidad": 1},
             {"ref": "R1", "cantidad": "dos"}, {"ref": "R1", "cantidad": 1}]:
    out = tools.dispatch("actualizar_carrito", args)
    print(args, "→", "ok" if out["ok"] else out["errors"])
store.close()

### 1.2 Control de afirmaciones
Con un LLM **simulado** que miente ("he actualizado el carrito" sin llamar a la herramienta), el código detecta la afirmación sin respaldo, pide corregir dos veces y, si persiste, responde con un mensaje de respaldo honesto. En las pruebas reales con Qwen2.5-7B este caso ocurrió y motivó este control.

In [ ]:
class LLMQueMiente:
    name = "simulado"
    def chat(self, messages, tools):
        return LLMReply("He actualizado el carrito con dos unidades.")

restore_store(paths.store_db, catalog.product_id)
store = Store(paths.store_db)
r = ShoppingAgent(LLMQueMiente(), store, retriever, catalog).turn("ponme dos")
print(format_turn(r))
print("Afirmación sin respaldo detectada:", r.unsupported_claim, "| pedidos:", store.summary()["orders"])
store.close()

## 2. Conversación real: compra con un problema de stock

Cinco turnos con el LLM real. Muestran la búsqueda con presupuesto, un stock insuficiente, el cambio de opción, la cotización, la confirmación y una confirmación repetida que no duplica el pedido.

In [ ]:
restore_store(paths.store_db, catalog.product_id)
store = Store(paths.store_db)
agent = ShoppingAgent(llm, store, retriever, catalog)

for msg in ["Hola, busco unos auriculares bluetooth. Tengo como máximo 60 euros.",
            "Quiero dos del primero que me dijiste.",
            "Vale, entonces dame dos de la tercera opción.",
            "Sí, confirmo la compra.",
            "Confirmo otra vez por si acaso."]:
    print(format_turn(agent.turn(msg)), end="\n\n")

In [ ]:
# Revisar el estado final: lo que el sistema recuerda y lo que quedó en la tienda.
print(agent.state.render(store, catalog.set_index("product_id").title))
print("\nTienda:", store.summary())
store.ops()[["seq", "op", "ok"]].tail(8)

## 3. Solicitud ambigua: el agente debe preguntar

"Quiero una funda" no dice para qué dispositivo. Lo esperado es una pregunta de aclaración **antes** de agregar nada al carrito.

In [ ]:
restore_store(paths.store_db, catalog.product_id)
store = Store(paths.store_db)
agent = ShoppingAgent(llm, store, retriever, catalog)
r = agent.turn("Quiero comprar una funda.")
print(format_turn(r))
print("\n¿Agregó algo al carrito?", any(c["name"] == "actualizar_carrito" for c in r.tool_calls))
store.close()

### Resumen y observaciones

- Las garantías críticas (no comprar sin confirmación, sin duplicados, stock y presupuesto) **no dependen del LLM**: las aplica el código y se cumplen aunque el modelo se equivoque.
- La calidad conversacional sí depende del modelo. En pruebas preliminares en la Mac, sobre la misma conversación:
  - Qwen2.5-7B no registró el presupuesto y respondió con listas.
  - Qwen3-8B tampoco registró el presupuesto y pronunció las refs.
  - Qwen3.5-9B lo hizo todo bien, pero es multimodal y su carga en Colab está por validar.
- **Siguiente:** `05_escenarios.ipynb`, con los 20 escenarios reproducibles, verificaciones automáticas del estado final y la comparación de modelos para justificar la elección.